# Cohorte Cancer - Femme (Controlled Tier Dataset v8)

## With Breast Cancer (BC)

In [ ]:
import pandas as pd

name_of_file_in_bucket = "df_final_cohort.tsv"

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
df_final_cohort = pd.read_csv(name_of_file_in_bucket)

In [ ]:
df_final_cohort["has_BC"].value_counts()

# European cohort

## Selection of samples

In [ ]:
df_rye = pd.read_csv("aou_admixture_estimates_rye_v8.Q", sep="\t")

In [ ]:
list_eur = df_rye[df_rye["eur"] >= 0.8]["research_id"].to_list()

In [ ]:
df_eur = df_final_cohort[df_final_cohort["B"].isin(list_eur)].copy()

In [ ]:
df_eur["#IID"] = df_eur["B"]

In [ ]:
keep_ids_eur = df_eur[["#IID","B"]].rename(columns={"B":"IID"})

In [ ]:
keep_ids_eur.to_csv("keep_ids_eur.txt", sep="\t", index=False)

In [ ]:
pheno_bc_eur = df_eur[['B', 'has_BC']].rename(columns={"B":"#IID"})

In [ ]:
pheno_bc_eur.value_counts('has_BC')

In [ ]:
pheno_bc_eur.to_csv("pheno_bc_eur.csv", sep="\t", index=False)

## Processing of genetic data

In [ ]:
# Selection of european sample for chr10
!plink2 --pfile acaf_threshold.chr10 --keep keep_ids_eur.txt --make-pgen --out filtered_eur_chr10

## Obtaining allele frequencies

In [ ]:
# CASES
!plink2 \
--pfile filtered_eur_chr10 \
--pheno pheno_bc_eur.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 1 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out cases_eur

In [ ]:
# CONTROLS
!plink2 \
--pfile filtered_eur_chr10 \
--pheno pheno_bc_eur.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 0 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out control_eur

## Frequency analysis

In [ ]:
pd.read_csv("cases_eur.afreq", sep="\t")

In [ ]:
pd.read_csv("control_eur.afreq", sep="\t")

## GWAS

In [ ]:
!plink2 \
    --pfile filtered_eur_chr10 \
    --pheno pheno_bc_eur.csv \
    --pheno-name has_BC \
    --1 \
    --glm hide-covar single-prec-cc allow-no-covars \
    --chr 10 --from-bp 121580593 --to-bp 121580593 \
    --threads 8 \
    --out filtered_eur_chr10_gwas

## GWAS Result

In [ ]:
pd.read_csv("filtered_eur_chr10_gwas.has_BC.glm.logistic.hybrid", sep="\t")

# African cohort

## Selection samples

In [ ]:
list_afr = df_rye[df_rye["afr"] >= 0.8]["research_id"].to_list()

In [ ]:
df_afr = df_final_cohort[df_final_cohort["B"].isin(list_afr)].copy()

In [ ]:
df_afr["#IID"] = df_afr["B"]

In [ ]:
keep_ids_afr = df_afr[["#IID","B"]].rename(columns={"B":"IID"})

In [ ]:
keep_ids_afr.to_csv("keep_ids_afr.txt", sep="\t", index=False)

In [ ]:
pheno_bc_afr = df_afr[['B', 'has_BC']].rename(columns={"B":"#IID"})

In [ ]:
pheno_bc_afr.value_counts('has_BC')

In [ ]:
pheno_bc_afr.head()

In [ ]:
pheno_bc_afr.to_csv("pheno_bc_afr.csv", sep="\t", index=False)

## Processing of genetic data

In [ ]:
!plink2 --pfile acaf_threshold.chr10 --keep keep_ids_afr.txt --make-pgen --out filtered_afr_chr10

## Obtaining allele frequencies

In [ ]:
!plink2 \
--pfile filtered_afr_chr10 \
--pheno pheno_bc_afr.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 1 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out cases_afr

In [ ]:
!plink2 \
--pfile filtered_afr_chr10 \
--pheno pheno_bc_afr.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 0 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out control_afr

## Frequency analysis

In [ ]:
pd.read_csv("cases_afr.afreq", sep="\t")

In [ ]:
pd.read_csv("control_afr.afreq", sep="\t")

## GWAS

In [ ]:
!plink2 \
    --pfile filtered_afr_chr10 \
    --pheno pheno_bc_afr.csv \
    --pheno-name has_BC \
    --1 \
    --glm hide-covar single-prec-cc allow-no-covars \
    --chr 10 --from-bp 121580593 --to-bp 121580593 \
    --threads 8 \
    --out filtered_afr_chr10_gwas

## GWAS Result

In [ ]:
pd.read_csv("filtered_afr_chr10_gwas.has_BC.glm.logistic.hybrid", sep="\t")